# Eye-state classifier branch (deep learning)

Same job as `EAR_Detection.ipynb` (PERCLOS, eye-closure time, blink rate, confidence), but the open/closed decision comes from a **neural network that looks at the eye picture** instead of landmark geometry.
So the two branches fail in different situations, which is what makes fusing them useful.

```
frame -> face detector (YuNet: box + eye centres) -> two eye crops -> CNN -> p(closed) per eye
      -> state OPEN / CLOSED / UNKNOWN -> rolling 30 s window -> PERCLOS, closure time, blinks, score, confidence
```
The notebook goes in order: **1. image, 2. video file, 3. live camera**. Window statistics (closures, PERCLOS, score) come from `drowsiness/common/windowing.py`, the same rules as the EAR notebook.

## 0. Setup

In [ ]:
# Run once in the project root, then pick the .venv kernel for this notebook:
# python3 -m venv .venv && .venv/bin/pip install -r requirements.txt
# python3 download_models.py          # YuNet face detector (and the old model, kept for comparison)
# python3 train_eye_model.py train small 80 14   # trains our eye model (about 10 minutes) -> models/eye_state_small_80.onnx

In [ ]:
import csv, json, os, time
from collections import deque

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from IPython.display import Image, display

from drowsiness.common.windowing import (
    DROWSY_CLOSURE_DURATION, MICROSLEEP_DURATION, channel_output)

IMAGE_PATH = "data/test2.jpg"          # your own test image (not in git)
VIDEO_PATH = "data/test_vid_1.mov"     # your own test video (not in git)
FACE_MODEL = "models/face_detection_yunet_2023mar.onnx"
EYE_MODEL = "models/eye_state.onnx"

for path in (FACE_MODEL, EYE_MODEL):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} is missing: see the commands at the top of this notebook")

## 1. Which model, and why

The old OpenVINO model was not used; four candidates were measured on the same eye images (script `train_eye_model.py compare`, full table in `docs/eye_model_comparison.md`).
Two test sets: **MRL** = 6000 infrared eye images of 8 people the model never trained on, **webcam** = 206 colour crops from your video (one person only, so just a sanity check).
`false open` = a closed eye called open (the dangerous mistake).

In [ ]:
results = json.load(open("docs/eye_model_comparison.json"))
rows = []
for name, r in results.items():
    if name.endswith("_ms"):
        continue
    rows.append({
        "model": name, "params": r["params"], "licence": r["licence"],
        "MRL accuracy %": 100 * r["mrl"]["accuracy"], "MRL false open %": 100 * r["mrl"]["false_open_rate"],
        "MRL false closed %": 100 * r["mrl"]["false_closed_rate"],
        "webcam accuracy %": 100 * r["webcam"]["accuracy"], "CPU ms per eye": r["ms_per_eye_cpu"]})
pd.DataFrame(rows).round(1).set_index("model")

## 2. Face detector, eye crops, classifier

* **Face + eye centres: YuNet** (OpenCV, MIT licence). It returns the face box and both eye centres, finds the face in more frames than the Haar detector, and does **not** need dlib landmarks. So if the EAR branch loses its landmarks, this branch still works.
* **Eye crop:** a square around each eye centre, side = 0.65 x the distance between the eyes. That framing (eye + brow) is what the training images look like.
* **Classifier:** MobileNetV3 fine-tuned on the MRL Eye dataset (84,898 eye images, 37 people, split by person), with strong augmentation (light, blur, noise, rotation, mirror). Input: grey 80x80 px eye, output: probability the eye is closed. Runs in about 1 ms with ONNX Runtime.

In [ ]:
face_detector = cv2.FaceDetectorYN.create(FACE_MODEL, "", (320, 320), 0.6, 0.3, 5000)
eye_session = ort.InferenceSession(EYE_MODEL, providers=["CPUExecutionProvider"])
EYE_SIZE = eye_session.get_inputs()[0].shape[-1]
print(f"eye model input: {EYE_SIZE}x{EYE_SIZE} grey, providers: {eye_session.get_providers()}")

DETECT_WIDTH = 640     # big frames are shrunk to this width for face detection only (speed)
CROP_SCALE = 0.65      # eye crop side = this x distance between the eyes

In [ ]:
def detect_face(frame):
    """Largest face as a dict (box, score, eyes, nose in full-frame pixels), or None."""
    scale = min(1.0, DETECT_WIDTH / frame.shape[1])
    small = cv2.resize(frame, None, fx=scale, fy=scale) if scale < 1 else frame
    face_detector.setInputSize((small.shape[1], small.shape[0]))
    _, faces = face_detector.detect(small)
    if faces is None:
        return None
    f = max(faces, key=lambda r: r[2] * r[3]).copy()
    f[:14] /= scale
    # YuNet order: box(4), right eye, left eye, nose, mouth corners (x, y each), score
    return {"box": f[:4].astype(int), "score": float(f[14]), "eyes": np.array([f[4:6], f[6:8]]), "nose": f[8:10]}


def crop_eye(frame, centre, side):
    """Square crop around an eye centre -> (crop or None, share of the square that is inside the frame)."""
    h, w = frame.shape[:2]
    x0, y0 = int(round(centre[0] - side / 2)), int(round(centre[1] - side / 2))
    x1, y1 = x0 + int(round(side)), y0 + int(round(side))
    cx0, cy0, cx1, cy1 = max(x0, 0), max(y0, 0), min(x1, w), min(y1, h)
    if cx1 - cx0 < 4 or cy1 - cy0 < 4:
        return None, 0.0
    return frame[cy0:cy1, cx0:cx1], (cx1 - cx0) * (cy1 - cy0) / ((x1 - x0) * (y1 - y0))


def predict_closed(crops):
    """Probability that each eye crop is closed (one batch for both eyes)."""
    batch = np.stack([cv2.resize(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY), (EYE_SIZE, EYE_SIZE), interpolation=cv2.INTER_AREA)
                      for c in crops]).astype(np.float32)[:, None] / 255
    return eye_session.run(None, {"eye": batch})[0]

## 3. Confidence: how much can this frame be trusted?

Confidence (0-1) is the **minimum** of eight simple factors, so any one problem lowers it (a chain is as strong as its weakest link). No face gives 0.

| factor | drops when |
|---|---|
| `face` | the face detector is unsure it is a face |
| `size` | the eyes are only a few pixels wide (far away, low resolution) |
| `pose` | the head is turned (nose off centre) or strongly tilted |
| `inside` | an eye crop falls partly outside the image |
| `light` | the eyes are too dark or over-exposed |
| `agree` | the two eyes disagree strongly (glare on glasses, hand or hair over one eye) |
| `certain` | the network is undecided (probability near 0.5): no hard verdict from an unclear eye |
| `stable` | the eye positions jump between frames (tracking is shaky) |

Frames below `MIN_CONF` become `UNKNOWN`: they are never counted as open or closed, and PERCLOS ignores them.

In [ ]:
MIN_CONF = 0.3          # below this a frame is "UNKNOWN", not OPEN/CLOSED
CLOSED_ON = 0.5         # p(closed) above this: eyes become CLOSED
CLOSED_OFF = 0.4        # an already CLOSED eye must fall below this to be OPEN again (stops flicker)


def clip01(x):
    return float(np.clip(x, 0.0, 1.0))


def input_confidence(face, eyes, crops, inside, p, prev_eyes):
    """Return (confidence, factors). Each factor is in [0, 1]; confidence = min of them."""
    iod = np.linalg.norm(eyes[0] - eyes[1])                          # distance between the eyes
    side = CROP_SCALE * iod
    offset = abs(face["nose"][0] - eyes[:, 0].mean()) / iod          # nose off centre = head turned
    tilt = abs(np.degrees(np.arctan2(*(eyes[1] - eyes[0])[::-1])))
    tilt = min(tilt, 180 - tilt)
    brightness = np.mean([c.mean() for c in crops])

    factors = {
        "face": clip01((face["score"] - 0.6) / 0.3),
        "size": clip01((side - 16) / 24),                            # 16 px -> 0, 40 px -> 1
        "pose": min(1 - clip01((offset - 0.10) / 0.25), 1 - clip01((tilt - 20) / 30)),
        "inside": clip01((min(inside) - 0.6) / 0.4),
        "light": min(clip01((brightness - 35) / 35), clip01((240 - brightness) / 30)),
        "agree": 1 - clip01((abs(p[0] - p[1]) - 0.3) / 0.5),
        "certain": clip01((max(p.mean(), 1 - p.mean()) - 0.5) / 0.2),      # an undecided network is not a verdict
        "stable": 1.0,
    }
    if prev_eyes is not None:                                        # eye shift since last frame, relative to iod
        shift = np.linalg.norm(eyes - prev_eyes, axis=1).mean() / iod
        factors["stable"] = 1 - clip01((shift - 0.05) / 0.25)
    return min(factors.values()), factors


def process_frame(frame, prev_eyes=None):
    """One frame: face -> eye crops -> CNN -> state and confidence. Never raises on a missing face."""
    res = {"face": None, "eyes": None, "side": 0.0, "p_left": np.nan, "p_right": np.nan, "p_closed": np.nan,
           "state": "UNKNOWN", "confidence": 0.0, "factors": {}, "ms_face": 0.0, "ms_cnn": 0.0}

    t0 = time.perf_counter()
    face = detect_face(frame)
    res["ms_face"] = 1000 * (time.perf_counter() - t0)
    if face is None:
        return res
    res["face"] = face
    res["eyes"] = face["eyes"]
    iod = np.linalg.norm(face["eyes"][0] - face["eyes"][1])
    res["side"] = CROP_SCALE * iod

    cropped = [crop_eye(frame, centre, res["side"]) for centre in face["eyes"]]
    if iod < 8 or any(crop is None for crop, _ in cropped):
        return res                                                    # face found, eyes unusable -> UNKNOWN
    crops, inside = [c for c, _ in cropped], [i for _, i in cropped]

    t0 = time.perf_counter()
    p = predict_closed(crops)
    res["ms_cnn"] = 1000 * (time.perf_counter() - t0)

    res["confidence"], res["factors"] = input_confidence(face, face["eyes"], crops, inside, p, prev_eyes)
    res["p_right"], res["p_left"], res["p_closed"] = float(p[0]), float(p[1]), float(p.mean())
    if res["confidence"] >= MIN_CONF:
        res["state"] = "CLOSED" if res["p_closed"] > CLOSED_ON else "OPEN"
    return res


def stabilise(res, recent_p, last_state):
    """Steadier verdict for streams: p(closed) is averaged over 3 frames and an eye that is CLOSED
    needs p below CLOSED_OFF to count as OPEN again. Works for video files and the live camera."""
    if res["state"] == "UNKNOWN":
        recent_p.clear()
        return
    recent_p.append(res["p_closed"])
    res["p_closed"] = float(np.mean(recent_p))
    res["state"] = "CLOSED" if res["p_closed"] > (CLOSED_OFF if last_state == "CLOSED" else CLOSED_ON) else "OPEN"

## 4. Drawing: boxes, eye probabilities, alert

* Face box and eye squares are green (OPEN), red (CLOSED) or orange (UNKNOWN).
* **DROWSY** (eyes closed 1 s or more) and **MICROSLEEP** (2 s or more) show as a big red box under the face box. The levels and their sources are listed in `drowsiness/common/windowing.py`.

In [ ]:
STATE_COLOUR = {"OPEN": (0, 200, 0), "CLOSED": (0, 0, 255), "UNKNOWN": (0, 165, 255)}


def draw_hud(frame, res, out=None, note="", last_box=None):
    """Draw on `frame` in place: face box, eye squares with p(closed), values, and the alert."""
    height, width = frame.shape[:2]
    font = float(np.clip(width / 1100, 0.55, 1.4))                     # text grows with the picture
    thick = max(2, int(font * 2.5))
    colour = STATE_COLOUR[res["state"]]

    if res["face"] is not None:
        x, y, w, h = res["face"]["box"]
        cv2.rectangle(frame, (x, y), (x + w, y + h), colour, thick + 1)
        for centre, p in zip(res["eyes"], (res["p_right"], res["p_left"])):
            half = int(res["side"] / 2)
            cx, cy = int(centre[0]), int(centre[1])
            cv2.rectangle(frame, (cx - half, cy - half), (cx + half, cy + half), colour, thick)
            if not np.isnan(p):
                cv2.putText(frame, f"{p:.2f}", (cx - half, cy - half - 6), cv2.FONT_HERSHEY_SIMPLEX, font * 0.7, colour, thick)

    lines = [f"{res['state']}   p(closed) {res['p_closed']:.2f}" if not np.isnan(res["p_closed"]) else "NO FACE FOUND - confidence 0, no verdict",
             f"CONFIDENCE frame {res['confidence']:.2f}" + (f"  channel {out['confidence']:.2f}" if out else "")]
    if out is not None:
        r = out["raw"]
        lines += [f"PERCLOS {r['perclos']:.2f}  blinks {r['blink_count']} ({r['blink_rate_per_min']:.0f}/min)",
                  f"closed now {r['current_closure_s']:.1f}s  longest {r['longest_closure_s']:.1f}s",
                  f"SCORE {out['score']:.2f}"]
    if res["state"] == "UNKNOWN" and res["face"] is not None:
        lines.append("EYES NOT CLEARLY VISIBLE - low confidence, no verdict")
    if res["factors"]:
        weakest = min(res["factors"], key=res["factors"].get)
        lines.append(f"weakest factor: {weakest} {res['factors'][weakest]:.2f}")
    if note:
        lines.append(note)
    for i, text in enumerate(lines):
        pos = (10, int(26 * font) + int(26 * font) * i)
        cv2.putText(frame, text, pos, cv2.FONT_HERSHEY_SIMPLEX, font * 0.7, (0, 0, 0), thick + 2)
        cv2.putText(frame, text, pos, cv2.FONT_HERSHEY_SIMPLEX, font * 0.7, (255, 255, 255), thick)

    # DROWSY / MICROSLEEP: big red alert right below the face box
    level = out["raw"]["closure_level"] if out is not None else "NONE"
    box = res["face"]["box"] if res["face"] is not None else last_box
    if level in ("DROWSY", "MICROSLEEP") and box is not None:
        x, y, w, h = box
        text = f"{level} {out['raw']['current_closure_s']:.1f}s"
        (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font * 1.6, thick * 2)
        top = min(y + h + 10, height - th - base - 26)
        cv2.rectangle(frame, (x, top), (x + max(w, tw + 20), top + th + base + 16), (0, 0, 255), -1)
        cv2.putText(frame, text, (x + 10, top + th + 8), cv2.FONT_HERSHEY_SIMPLEX, font * 1.6, (255, 255, 255), thick * 2)

## 5. Step 1: a single image

In [ ]:
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

res = process_frame(image)
if res["face"] is None:
    print("No face found in the image")
else:
    print(f"state {res['state']}   p(closed): right eye {res['p_right']:.3f}, left eye {res['p_left']:.3f}")
    print(f"confidence {res['confidence']:.2f}   factors {({k: round(v, 2) for k, v in res['factors'].items()})}")
    print(f"face detector {res['ms_face']:.1f} ms, classifier {res['ms_cnn']:.1f} ms")

    shown = image.copy()
    draw_hud(shown, res)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [3, 1, 1]})
    axes[0].imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
    for ax, centre, p, title in zip(axes[1:], res["eyes"], (res["p_right"], res["p_left"]), ("right eye", "left eye")):
        crop, _ = crop_eye(image, centre, res["side"])
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.set_title(f"{title}: p(closed) {p:.2f}")
    for ax in axes:
        ax.axis("off")
    plt.show()

## 6. Steps 2 and 3: video file and live camera (one function)

`run_stream(source)` takes a **video path, a camera index (0 = laptop camera) or a stream URL** and calculates everything **for every frame**, over a rolling 30 s window:
PERCLOS, current and longest eye-closure time, blink count and rate, drowsiness score, and the **confidence** of the channel (mean frame confidence, capped by the last half second, so it falls at once when the face or eyes are lost).

* `show="notebook"` shows frames inline, `"window"` opens an OpenCV window (press `q`), `None` shows nothing.
* `save_to="runs/x.mp4"` writes the annotated video, `log_csv="runs/x.csv"` writes one row per frame.
* No calibration is needed: the network decides from the picture, not from a person-specific threshold.
* **macOS camera:** the app that runs this notebook (Terminal, VS Code, Jupyter) needs permission under System Settings > Privacy & Security > Camera.

In [ ]:
def run_stream(source=0, window_s=30.0, show="notebook", save_to=None, log_csv=None, max_seconds=None, display_width=960):
    """Run the branch on a video file, a camera index or a stream URL. Returns a history dict
    (arrays per frame) with the last channel output under "output"."""
    if isinstance(source, str) and source.isdigit():
        source = int(source)
    is_camera = isinstance(source, int)

    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open source {source!r}. For a camera: allow camera access for this app in "
                           "System Settings > Privacy & Security > Camera, and check the index (0, 1, ...).")

    window = {"t": deque(), "state": deque(), "conf": deque()}
    hist = {k: [] for k in ("t", "p_closed", "state", "confidence", "perclos", "closure_s", "score", "channel_conf",
                            "ms_total", "ms_face", "ms_cnn")}
    recent_p, last_state, prev_eyes, last_box = deque(maxlen=3), "UNKNOWN", None, None
    output, handle, writer, log, log_file = None, None, None, None, None
    black_frames, fps_smooth, t_start = 0, 0.0, time.time()
    if log_csv:
        os.makedirs(os.path.dirname(log_csv) or ".", exist_ok=True)
        log_file = open(log_csv, "w", newline="")
        log = csv.writer(log_file)
        log.writerow(["time_s", "p_closed", "state", "frame_confidence", "perclos", "closure_s", "closure_level",
                      "blink_count", "score", "channel_confidence"])

    try:
        while True:
            t_loop = time.perf_counter()
            ok, frame = cap.read()
            if not ok:
                print("Stream ended or camera stopped delivering frames.")
                break

            black_frames = black_frames + 1 if frame.mean() < 2 else 0     # macOS permission problems give black frames
            if black_frames == 30:
                print("Camera frames are black: macOS camera permission may be missing for this app.")

            t = time.time() - t_start if is_camera else cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
            if max_seconds is not None and t > max_seconds:
                break

            res = process_frame(frame, prev_eyes)
            prev_eyes = res["eyes"]
            last_box = res["face"]["box"] if res["face"] is not None else last_box
            stabilise(res, recent_p, last_state)
            last_state = res["state"]

            window["t"].append(t); window["state"].append(res["state"]); window["conf"].append(res["confidence"])
            while t - window["t"][0] > window_s:
                for q in window.values():
                    q.popleft()
            output = channel_output(window["t"], window["state"], window["conf"], window_s)

            ms_total = 1000 * (time.perf_counter() - t_loop)
            fps_smooth = 1000 / ms_total if fps_smooth == 0 else 0.9 * fps_smooth + 0.1 * 1000 / ms_total
            r = output["raw"]
            for key, value in (("t", t), ("p_closed", res["p_closed"]), ("state", res["state"]), ("confidence", res["confidence"]),
                               ("perclos", r["perclos"]), ("closure_s", r["current_closure_s"]), ("score", output["score"]),
                               ("channel_conf", output["confidence"]), ("ms_total", ms_total), ("ms_face", res["ms_face"]),
                               ("ms_cnn", res["ms_cnn"])):
                hist[key].append(value)
            if log:
                log.writerow([f"{t:.3f}", f"{res['p_closed']:.4f}", res["state"], f"{res['confidence']:.3f}", f"{r['perclos']:.4f}",
                              f"{r['current_closure_s']:.3f}", r["closure_level"], r["blink_count"], f"{output['score']:.3f}",
                              f"{output['confidence']:.3f}"])

            draw_hud(frame, res, output, f"{fps_smooth:.0f} FPS, {ms_total:.0f} ms", last_box)

            scale = min(1.0, display_width / frame.shape[1])                # saved / shown picture is smaller for speed
            small = cv2.resize(frame, None, fx=scale, fy=scale) if scale < 1 else frame

            if save_to and writer is None:
                os.makedirs(os.path.dirname(save_to) or ".", exist_ok=True)
                video_fps = cap.get(cv2.CAP_PROP_FPS)
                writer = cv2.VideoWriter(save_to, cv2.VideoWriter_fourcc(*"mp4v"),
                                         video_fps if not is_camera and video_fps > 0 else 30, (small.shape[1], small.shape[0]))
            if writer is not None:
                writer.write(small)

            if show == "window":
                cv2.imshow("eye-state CNN (q to quit)", small)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
            elif show == "notebook":
                picture = Image(data=cv2.imencode(".jpg", small)[1].tobytes())
                if handle is None:
                    handle = display(picture, display_id=True)
                else:
                    handle.update(picture)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        cap.release()
        if writer is not None:
            writer.release()
        if log_file:
            log_file.close()
        if show == "window":
            cv2.destroyAllWindows()

    hist = {k: np.array(v) for k, v in hist.items()}
    hist["output"] = output
    return hist

### 6a. Your video file

In [ ]:
video = run_stream(VIDEO_PATH, show=None, save_to="runs/dl_video_annotated.mp4", log_csv="runs/dl_video_log.csv")

print(f"{len(video['t'])} frames, {video['t'][-1]:.1f} s")
print(f"speed: {np.mean(video['ms_total']):.1f} ms per frame (face detector {np.mean(video['ms_face']):.1f} ms, "
      f"classifier {np.mean(video['ms_cnn']):.1f} ms) = {1000 / np.mean(video['ms_total']):.0f} FPS")
final = video["output"]
print(f"\nscore {final['score']:.2f} (0 = alert, 1 = drowsy)    channel confidence {final['confidence']:.2f}")
for key, value in final["raw"].items():
    print(f"  {key}: {value}")
print("\nannotated video: runs/dl_video_annotated.mp4   per-frame log: runs/dl_video_log.csv")

The values below are calculated **for every frame** over the rolling 30 s window (blue = this notebook's CNN branch). Grey bands: frames with no usable face (`UNKNOWN`).

In [ ]:
t = video["t"]
unknown = video["state"] == "UNKNOWN"
fig, axes = plt.subplots(5, 1, figsize=(14, 13), sharex=True)
panels = [("p(closed), both eyes", video["p_closed"], (-0.05, 1.05)),
          ("current eye-closure time (s)", video["closure_s"], None),
          ("PERCLOS (rolling window)", video["perclos"], (-0.05, 1.05)),
          ("drowsiness score (0 alert, 1 drowsy)", video["score"], (-0.05, 1.05)),
          ("confidence: frame (light) and channel (dark)", video["channel_conf"], (-0.05, 1.05))]
for ax, (title, values, ylim) in zip(axes, panels):
    ax.plot(t, values, color="tab:blue"); ax.set_ylabel(title, fontsize=8); ax.grid(alpha=0.3)
    if ylim: ax.set_ylim(*ylim)
    ax.fill_between(t, 0, 1, where=unknown, transform=ax.get_xaxis_transform(), color="grey", alpha=0.2)
axes[0].axhline(CLOSED_ON, ls="--", color="k", lw=0.8)
axes[1].axhline(DROWSY_CLOSURE_DURATION, ls="--", color="orange", lw=0.8); axes[1].axhline(MICROSLEEP_DURATION, ls="--", color="red", lw=0.8)
axes[4].plot(t, video["confidence"], color="tab:blue", alpha=0.3)
axes[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

### 6b. Live camera
Starts your laptop camera (`0`; try `1` for another camera, or a video path / stream URL). Interrupt the kernel (stop button) to end it, or set `max_seconds`.

In [ ]:
live = run_stream(0, show="notebook", log_csv="runs/dl_live_log.csv")
print(live["output"])